In [1]:
import pandas as pd
import duckdb

df_device_alarm_daily = pd.DataFrame({
    "device_id": [
        "R05", "R05", "R05", "R05",
        "R16", "R16", "R16", "R16",
        "R34", "R34", "R34", "R34"
    ],
    "stat_date": [
        "2026-07-01",
        "2026-07-02",
        "2026-07-03",
        "2026-07-05",
        "2026-07-01",
        "2026-07-02",
        "2026-07-04",
        "2026-07-05",
        "2026-07-01",
        "2026-07-02",
        "2026-07-03",
        "2026-07-04"
    ],
    "alarm_count": [
        2, 0, 3, 1,
        1, 2, 0, 4,
        0, 1, 2, 1
    ]
})

df_device_alarm_daily["stat_date"] = pd.to_datetime(
    df_device_alarm_daily["stat_date"]
)

df_device_alarm_daily

,device_id,stat_date,alarm_count
0,R05,2026-07-01,2
1,R05,2026-07-02,0
2,R05,2026-07-03,3
3,R05,2026-07-05,1
4,R16,2026-07-01,1
5,R16,2026-07-02,2
6,R16,2026-07-04,0
7,R16,2026-07-05,4
8,R34,2026-07-01,0
9,R34,2026-07-02,1


# SQL Daily Review：设备累计告警次数

## 题目背景

设备每天会汇总当日产生的告警次数。

现在需要按照日期顺序，计算每台设备截至当天的累计告警次数。

表中同一台设备、同一天最多只有一条记录。

## 题目要求

为每条设备日统计记录计算累计告警次数。

累计范围为：

```text
同一设备从最早记录日期开始，一直累计到当前记录日期
```

### 输出字段

| 字段 | 含义 |
|---|---|
| `device_id` | 设备编号 |
| `stat_date` | 统计日期 |
| `alarm_count` | 当日告警次数 |
| `cumulative_alarm_count` | 截至当天的累计告警次数 |

### 计算规则

每台设备分别累计，不能把不同设备的数据混在一起。

例如，设备某几天的告警次数为：

```text
2、0、3、1
```

对应的累计告警次数应为：

```text
2、2、5、6
```

日期缺失不代表告警次数为 `0`。

例如存在：

```text
2026-07-03
2026-07-05
```

只能说明表中没有 `2026-07-04` 的记录，不需要自动补齐日期。

### 最终排序

按照以下顺序排列：

1. `device_id` 升序；
2. `stat_date` 升序。

## 解题要求

- 使用窗口函数 `SUM()`；
- 使用 `PARTITION BY` 按设备分别累计；
- 使用 `ORDER BY` 按日期确定累计顺序；
- 不使用相关子查询；
- 保留原始明细行，不使用 `GROUP BY` 汇总成一行。

In [3]:
query = '''

    SELECT
        device_id,
        stat_date,
        alarm_count,
        SUM(alarm_count)
        OVER(
            PARTITION BY device_id ORDER BY stat_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ):: INTEGER AS cumulative_alarm_count
    FROM df_device_alarm_daily
    ORDER BY device_id,stat_date
'''
df = duckdb.execute(query).fetchdf()
df

,device_id,stat_date,alarm_count,cumulative_alarm_count
0,R05,2026-07-01,2,2
1,R05,2026-07-02,0,2
2,R05,2026-07-03,3,5
3,R05,2026-07-05,1,6
4,R16,2026-07-01,1,1
5,R16,2026-07-02,2,3
6,R16,2026-07-04,0,3
7,R16,2026-07-05,4,7
8,R34,2026-07-01,0,0
9,R34,2026-07-02,1,1
